# Named-entity Recognition (NER) 

Named entity recognition is a fundamental task in information extraction from textual documents. While named entities originally corresponded to real-world entities with names (named entities), this concept has been extended to any type of information: it is possible to extract chemical molecules, product numbers, amounts, addresses, etc. In this practical assignment, we will use several named entity extraction libraries in French on a small corpus. The objective is not to train the best possible model, but to test the use of each of these libraries.



## The AdminSet dataset
The AdminSet dataset is a corpus of administrative documents in French produced by automatic character recognition and manually annotated with named entities. This corpus is quite difficult because the document recognition process produces noisy text (errors due to layout, recognition, fonts, etc.).

The paper describing the dataset is available [here](https://hal.science/hal-04855066v1/file/AdminSet_et_AdminBERT__version___preprint.pdf).

The corpus is available on HuggingFace: [Adminset-NER](https://huggingface.co/datasets/taln-ls2n/Adminset-NER).

In [2]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 36.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.3 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [datasets]/13 [datasets]ce-hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.10.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset
ds = load_dataset('taln-ls2n/Adminset-NER')
print(ds)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 85/85 [00:00<00:00, 21609.64 examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 729
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 85
    })
})


#### Question
> * Compute descriptive statistics on the texts  for each split (train, dev)
> * Compute descriptive statistics on the entities for each split (train, dev)
> * Compare with the statistics reported in the paper (Table 2)
> * Display a couple of random texts with their entities

In [5]:
from datasets import load_dataset
import numpy as np
from collections import Counter
import pandas as pd

# Chargement du dataset Adminset-NER depuis HuggingFace
ds = load_dataset('taln-ls2n/Adminset-NER')

def get_stats(dataset_split):
    # Calcul de la longueur de chaque document (nombre de tokens)
    lengths = [len(x['tokens']) for x in dataset_split]
    
    # Extraction de tous les labels d'entités (différents de 'O')
    all_entities = [label for sublist in dataset_split['ner_tags'] for label in sublist if label != 0]
    
    stats = {
        "count": len(lengths), #nb total de documents
        "min": np.min(lengths), #longueur minimale, le texte le plus court de toute ma gang
        "max": np.max(lengths), #texte le plus long
        "mean": np.mean(lengths), #moyenne
        "std": np.std(lengths),
        "median": np.median(lengths),
        "total_entities": len(all_entities) #C'est le compte total des entités nommées qu'on a trouvées. On ne compte pas les mots ordinaires (les 'O' dans le format BIO), juste les vraies entités comme les noms de personnes, de lieux ou d'organisations
    }
    return stats

# Affichage des statistiques pour Train et Dev
for split in ['train', 'validation']:
    print(f"--- Statistiques pour {split} ---")
    print(get_stats(ds[split]))

# Visualisation d'un exemple aléatoire
import random
idx = random.randint(0, len(ds['train'])-1)
example = ds['train'][idx]
print(f"\nExemple de texte (index {idx}):\n", " ".join(example['tokens']))
print("Entités correspondantes:", example['ner_tags'])

--- Statistiques pour train ---
{'count': 729, 'min': np.int64(15), 'max': np.int64(379), 'mean': np.float64(63.3676268861454), 'std': np.float64(52.66968106175962), 'median': np.float64(45.0), 'total_entities': 46195}
--- Statistiques pour validation ---
{'count': 85, 'min': np.int64(19), 'max': np.int64(352), 'mean': np.float64(79.83529411764705), 'std': np.float64(68.27381671214015), 'median': np.float64(50.0), 'total_entities': 6786}

Exemple de texte (index 703):
 9 < page > 9 < / page > Ses compétences en valorisation du patrimoine géologique , en médiation des sciences de la terre et plus largement en animation culturelle du territoire , font de la Maison des Minéraux un équipement reconnu et incontournable .
Entités correspondantes: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O']


### Creation of the splits

The train_test_split() function from huggingface allow to split a dataset randomly in 2 parts : https://huggingface.co/docs/datasets/v4.5.0/process#split

The ```spacy_utils.py``` file contains functions to save a dataset in text format (```save_text```, usefull for inspection), BIO format (```save_bio```) and spacy format (```save_docbin```).

#### Questions
>* Using the split function, create a train/dev/test split corresponding to the proportions reported in the paper
>* Save the sets in a corpus directory, in text, bio and docbin formats.

In [7]:
pip install spacy_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 3.6 MB/s  0:00:09m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.9/835.9 kB 10.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 606.2 kB/s  0:00:18m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [spacy_utils] [spacy]]thlib]
Note: you may need to restart the kernel to use updated packages.


In [10]:
# On check c'est quoi les clés qu'on a pour être sûr
print("Tes clés actuelles :", ds.keys())

# Si 'test' est pas là, on va le créer à partir du set de validation
# On prend une partie du validation pour en faire notre test
if 'test' not in ds:
    print("Oups, il manque le test! On va arranger ça.")
    
    # On split le set de validation en deux (50/50 pour le dev et le test)
    # C'est ici qu'on utilise la fonction demandée dans ton TP
    split_dev_test = ds['validation'].train_test_split(test_size=0.5, seed=42)
    
    # On remonte notre dictionnaire au complet
    from datasets import DatasetDict
    ds = DatasetDict({
        'train': ds['train'],
        'validation': split_dev_test['train'], # Ça devient notre nouveau 'dev'
        'test': split_dev_test['test']        # Pis v'là notre set de test!
    })

print("Là on est en business! Nouvelles clés :", ds.keys())

Tes clés actuelles : dict_keys(['train', 'validation'])
Oups, il manque le test! On va arranger ça, mon lapin.
Là on est en business! Nouvelles clés : dict_keys(['train', 'validation', 'test'])


In [11]:
import pandas as pd

def get_full_stats(dataset_dict):
    all_stats = []
    for split in dataset_dict.keys():
        # Longueur des textes (nombre de tokens)
        lengths = [len(x['tokens']) for x in dataset_dict[split]]
        # On compte les vraies entités (on ignore le 'O') [cite: 170]
        ents = [tag for sublist in dataset_dict[split]['ner_tags'] for tag in sublist if tag != 0]
        
        all_stats.append({
            "Split": split,
            "Nb Docs": len(lengths),
            "Moy. Tokens": round(np.mean(lengths), 1),
            "Médiane": np.median(lengths),
            "Total Entités": len(ents)
        })
    return pd.DataFrame(all_stats)

# Checke bin ça, ça va sortir un beau petit tableau propre
print("--- Comparaison avec la Table 2 du papier ---")
print(get_full_stats(ds))

--- Comparaison avec la Table 2 du papier ---
        Split  Nb Docs  Moy. Tokens  Médiane  Total Entités
0       train      729         63.4     45.0          46195
1  validation       42         73.0     42.0           3066
2        test       43         86.5     62.0           3720


In [12]:
import os
from spacy_utils import save_bio, save_text, save_docbin

# On se crée un dossier 'corpus' pour pas que les fichiers traînent partout
if not os.path.exists("corpus"):
    os.makedirs("corpus")

# On enregistre chaque split dans les 3 formats demandés
for split in ds.keys():
    # 1. Format texte brut pour zieuter les données
    save_text(ds[split], f"corpus/{split}.txt")
    # 2. Format BIO pour les modèles standards [cite: 169]
    save_bio(ds[split], f"corpus/{split}.bio")
    # 3. Format DocBin pour spaCy [cite: 282]
    save_docbin(ds[split], f"corpus/{split}.spacy")

print("C'est fait! Tes fichiers sont triés pis bien serrés dans le dossier /corpus.")

Saving text to corpus/train.txt...


100%|██████████| 729/729 [00:00<00:00, 4584.19it/s]


Saved to corpus/train.txt
Saving BIO text to corpus/train.bio...


100%|██████████| 729/729 [00:00<00:00, 6835.18it/s]


Saved to corpus/train.bio
Creating corpus/train.spacy with 729 examples...


...

Saved to corpus/test.txt
Saving BIO text to corpus/test.bio...


100%|██████████| 43/43 [00:00<00:00, 4556.61it/s]

Saved to corpus/test.bio
Creating corpus/test.spacy with 43 examples...



100%|██████████| 43/43 [00:00<00:00, 1065.49it/s]

Saved to corpus/test.spacy
C'est fait! Tes fichiers sont triés pis bien serrés dans le dossier /corpus.


### Testing spaCy pre-trained NER models

spaCy comes with a several pretrained models for many languages. For French, 4 models are provided : https://spacy.io/models/fr

To apply a pretrained model to dataset, use : 
- ```nlp = spacy.load(MODEL_NAME)``` to load the model. You need to download it first with "spacy download MODEL_NAME"
- ```DocBin().from_disk()``` to load a dataset in spaCy format from the disk
- ```doc_bin.get_docs(nlp.vocab)``` to convert the dataset from binary to text format
- ```nlp(doc.text)```to apply the NER model to a text

To evaluate the prediction, you can use the spaCy [Scorer](https://spacy.io/api/scorer)
- ```scorer.score(examples)``` where examples is a list of spaCy ```Example(prediction, reference)````

#### Question

>* Using a spaCy pretrained model for French, evaluate its performace for NER prediction on the train, dev and test sets
>* Compare this model to results reported in the paper

In [14]:
pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [19]:
# à mettre dans le terminal : python -m spacy download fr_core_news_sm

SyntaxError: invalid syntax (184992405.py, line 1)

In [20]:
import spacy
from spacy.tokens import DocBin
from spacy.scorer import Scorer
from spacy.training import Example

# Chargement du modèle français
nlp = spacy.load("fr_core_news_sm")

def evaluate_model(spacy_file):
    doc_bin = DocBin().from_disk(spacy_file)
    gold_docs = list(doc_bin.get_docs(nlp.vocab))
    
    examples = []
    for gold_doc in gold_docs:
        # On applique le modèle sur le texte brut
        pred_doc = nlp(gold_doc.text)
        examples.append(Example(pred_doc, gold_doc))
    
    # Calcul des scores (Précision, Rappel, F1) 
    scorer = Scorer()
    scores = scorer.score(examples)
    return scores['ents_per_type']

# Évaluation sur le split de test
test_scores = evaluate_model("corpus/test.spacy")
print("Performance par type d'entité (Test):", test_scores)

Performance par type d'entité (Test): {'MISC': {'p': 0.0, 'r': 0.0, 'f': 0.0}, 'ORG': {'p': 0.09090909090909091, 'r': 0.11320754716981132, 'f': 0.10084033613445378}, 'LOC': {'p': 0.09523809523809523, 'r': 0.25806451612903225, 'f': 0.13913043478260867}, 'PER': {'p': 0.24444444444444444, 'r': 0.26506024096385544, 'f': 0.254335260115607}}


In [ ]:
Rapport d'évaluation des performances NER (Modèle pré-entraîné)
Ce rapport analyse les résultats obtenus sur l'ensemble de test du corpus AdminSet en utilisant un modèle spaCy pré-entraîné pour le français (fr_core_news_sm).

1. Définition des métriques
L'évaluation repose sur trois indicateurs clés calculés à partir de la matrice de confusion  :
Précision (P) : Proportion d'entités prédites par le modèle qui sont réellement correctes. Elle mesure la capacité à éviter les Faux Positifs 
.P = TP/TP + FP. 
Rappel (R) : Proportion d'entités réelles présentes dans le texte que le modèle a réussi à capturer. Elle mesure la capacité à éviter les oublis (Faux Négatifs) 
R = TP/TP + FN. 
F1-Score (F) : Moyenne harmonique de la précision et du rappel, offrant une mesure globale de la performance .F_1 = 2*PR/(P + R)

2. Analyse des résultats par entité
Les scores obtenus montrent une performance globalement faible, typique d'un modèle non adapté au domaine spécifique des documents administratifs bruités.
on voit que le f1 score des PER est le plus gaut

3. Interprétation des échecs
Plusieurs facteurs expliquent ces scores bas :Erreurs de transcription (OCR) : Le corpus AdminSet contient du texte bruité issu de la reconnaissance automatique de caractères. Si un mot est mal orthographié (ex: "Mairie" devient "Ma1rie"), le modèle pré-entraîné ne le reconnaît plus.
    Décalage de domaine : Le modèle fr_core_news_sm a été entraîné sur des articles de presse (type Wikipedia/News). 
    Le vocabulaire et la structure des documents administratifs sont trop différents de ses données d'apprentissage.
    Frontières erronées : Une entité peut être partiellement détectée ou avec un mauvais type (ex: une organisation classée en lieu), ce qui dégrade fortement le score en évaluation stricte 

### Training a custom spaCy model

The training of a cupstom spaCy NER model can be done both with the command line interface (cli) or in a python script. Using the cli is ususally more optimzed. All the configuration of the training is defined in a coniguration file, which is a good practice for documentation, tracing and reproducibility.

The configuration file can be generated on line using the [Quickstart](https://spacy.io/usage/training#quickstart)

<img src="images/spacy_quickstart.jpg" width="600" >

You can run the training process as a script using the train function (https://spacy.io/usage/training#api-train), specifying the configuration file and the directory in which to save the model as parameters. Once the training is complete, the best and last models are saved in the directory.

#### Question
> * Generate a training configuration file for a NER in French
> * Add the correct path to the training and dev sets generated previously
> * train a NER model
> * Evaluate the model on the train, dev et test sets. Compare to the results reported in the paper.

In [ ]:
# train the model
from spacy.cli.train import train


In [ ]:
# evaluate the model

### Zero-shot NER prediction with GLiNER


[GLiNER](https://github.com/fastino-ai/GLiNER2/tree/main)  is a library that provides models for zero-shot named entity recognition. This means that[structured information extraction](https://github.com/fastino-ai/GLiNER2/blob/main/tutorial/3-json_extraction.md)structured information extraction, which means that the extracted information can be organised in a structured JSON format. GLiNER does not provide the location of entities in the text by default, but you can configure the model to output this information (```include_spans=True```). Finally, GLiNER enables entities to be overlapped and nested, which is not supported by the spaCy scorer. The spaCy [filter_spans](https://spacy.io/api/top-level#util.filter_spans) function can be used to remove overlapping entities for evaluation.

#### Question
> * Define the entities to extract from the text.
> * Apply GLiNER on the dev and test sets
> * Evaluate the models on the dev and test sets and compare to the results reported in the paper.

In [ ]:
from gliner2 import GLiNER2
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
from spacy.util import filter_spans
nlp = spacy.blank("fr")  # tokenizer only

doc_bin = DocBin().from_disk("REPLACE")
gold_docs = list(doc_bin.get_docs(nlp.vocab))

label_map = {
    # Define the entities here
}

gliner_labels = list(label_map.values())
reverse_map = {v: k for k, v in label_map.items()}

examples = []

for gold_doc in gold_docs:
    text = gold_doc.text
    
    predictions = extractor.extract_entities(text, gliner_labels, include_spans=True)
    pred_doc = nlp.make_doc(text)

    spans = []
    for gliner_label, entities in predictions["entities"].items():
        spacy_label = reverse_map.get(gliner_label)
        if not spacy_label:
            continue
        for ent in entities:
            start = ent["start"]
            end = ent["end"]

            span = pred_doc.char_span(start, end, label=spacy_label)
            if span:
                spans.append(span)

    spans = filter_spans(spans)
    pred_doc.ents = spans
    examples.append(Example(pred_doc, gold_doc))

